# Week 7: An introduction to pandas

So far we have worked with tables by writing our own loops: a loop to read every row, a loop to count values, a loop to add things up. That's a great way to understand what is happening, but it takes a lot of code, and we keep writing the same patterns over and over.

**pandas** is a Python library made for working with tables. It does the looping for us, so a question that took ten lines of code can often be answered in one. In this notebook we'll use pandas to revisit a question from the Week 7 notes: **where were the Giller nominees born, and where do they live now?**

### Our data

We'll use the file `giller_authors.csv`, which we saved at the end of the Week 7 notes. It is the table we cleaned by hand this week:
- It has **one row per author**, not one row per book. (Remember that this is a decision! A table of books would count Margaret Atwood four times.)
- The regions have already been cleaned with our corrections dictionary.
- Unknown values are written as `Unknown`.

Because the cleaning is already done, we can focus on what pandas can do.

## 1. Loading the data

`import pandas as pd` loads the library. Everyone abbreviates pandas as `pd`, so you'll see `pd.` at the start of many pandas commands.

`pd.read_csv()` reads a whole CSV file in one step. Compare this with the `with open(...)` and `csv.DictReader` code we have been writing!

In [ ]:
import pandas as pd

authors = pd.read_csv("giller_authors.csv")

print(len(authors), "authors")

pandas stores a table in a **DataFrame**. `.head()` shows us its first five rows.

In [ ]:
authors.head()

This looks a lot like a spreadsheet:
- The **column names** come from the header row of the CSV file.
- The numbers on the left (0, 1, 2, ...) are the **index**: pandas numbers the rows for us, starting from 0, just like a list.

## 2. Looking at one column

To get one column, we put its name in square brackets. This is just like getting a value from a dictionary, as in `book["Born"]`, except that we get the **whole column**: the value for every author at once.

A single column is called a **Series**.

In [ ]:
authors["Born region"]

pandas only shows the first and last few values of a long column, so the output stays readable.

## 3. Counting

Remember the counting loop we've written so many times?

```python
counts = {}
for author in birthplaces:
    region = ...
    if region in counts:
        counts[region] = counts[region] + 1
    else:
        counts[region] = 1
```

In pandas, that whole loop is one method: `.value_counts()`. It counts how many times each value appears in a column, and sorts the results from most to least common.

In [ ]:
authors["Born in Canada"].value_counts()

These are the same numbers we found in the Week 7 notes: 160 nominees born in Canada, 55 born elsewhere, and 3 unknown.

Now let's count the regions where the nominees were born:

In [ ]:
born_counts = authors["Born region"].value_counts()

born_counts

The result of `.value_counts()` is itself a Series, so we can save it in a variable, `born_counts`, and keep working with it. For example, `.head(10)` gives us just the first 10 values: the 10 most common regions.

In [ ]:
born_counts.head(10)

## 4. Our first chart

A list of numbers is precise, but a chart can show a pattern at a glance. pandas can draw a chart from a Series with `.plot`. Behind the scenes it uses another library called **matplotlib**, which we import as `plt`.

`.plot.barh()` draws a **horizontal bar chart**. Horizontal bars work well when the labels are words, because the labels can be written sideways and stay easy to read.

In [ ]:
import matplotlib.pyplot as plt

born_counts.head(10).plot.barh()
plt.show()

It works, but this chart isn't finished yet:
- The biggest bar is at the **bottom**. `barh` draws the first value at the bottom, so we flip it.
- There is **no title**, and the numbers along the bottom have **no label**. A reader shouldn't have to guess what the chart shows.

`.plot.barh()` gives back an object that we store in a variable called `ax` (short for "axes", matplotlib's word for one chart). We use `ax` to make these changes. A good title states the *finding*, not just the topic.

In [ ]:
ax = born_counts.head(10).plot.barh(color="#2a78d6")

ax.invert_yaxis()   # put the biggest bar at the top
ax.set_title("Most Giller nominees were born in Ontario")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")   # the region names speak for themselves
plt.show()

## 5. The same question for residences

Now we can ask where the nominees live, by changing just the column name. (We'll use orange for residences, so a reader can tell the two charts apart.)

In [ ]:
lives_counts = authors["Lives region"].value_counts()

ax = lives_counts.head(10).plot.barh(color="#eb6834")

ax.invert_yaxis()
ax.set_title("Most Giller nominees live in Ontario, and the USA is fourth")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")
plt.show()

**Discussion:** Compare the two charts. Which regions have more nominees living there than were born there? Which have fewer? What might explain the difference?

## 6. Choosing rows: filtering

So far we have always used every row. Often we only want some of them: for example, only the nominees who were born outside Canada. In pandas, choosing rows is called **filtering**, and it takes two steps.

**Step 1:** Write a condition about a column. Just as `==` compares two values, here it compares *every* value in the column, and gives back a column of `True` and `False`.

In [ ]:
authors["Born in Canada"] == "Elsewhere"

**Step 2:** Put that column of `True`/`False` inside square brackets after `authors`. pandas keeps only the rows where the value is `True`.

In [ ]:
born_elsewhere = authors[authors["Born in Canada"] == "Elsewhere"]

print(len(born_elsewhere), "authors born outside Canada")
born_elsewhere.head()

`born_elsewhere` is a new DataFrame with the same columns as `authors`, but only 55 rows. Everything we've learned works on it too. Where were these nominees born?

In [ ]:
ax = born_elsewhere["Born region"].value_counts().head(10).plot.barh(color="#2a78d6")

ax.invert_yaxis()
ax.set_title("Nominees born outside Canada come most often from the USA and England")
ax.set_xlabel("Number of authors")
ax.set_ylabel("")
plt.show()

Notice how we chained the steps together on one line: take the `"Born region"` column, count its values, keep the first 10, and plot them. You can always split a line like this into several steps with variables, as we did in section 3, if that's easier to read.

**Discussion:** The chart only shows the 10 most common places. How many different places are in the full list? (Try `len(born_elsewhere["Born region"].value_counts())`.) What does the chart hide about the variety of places Giller nominees come from?

Look closely at the bars after England: they are all tied at 2. Print `born_elsewhere["Born region"].value_counts()` and you'll find that other places (Ireland, for example) are tied at 2 as well but didn't make the top 10. pandas had to break the tie somehow. Is it fair to show some of these places and not others? How could you change the chart so that it doesn't make an arbitrary choice?

## 7. Try it yourself

1. Use `.value_counts()` to find out how many nominees live outside Canada.
2. Make a filtered DataFrame of the nominees who **live** outside Canada, and draw a chart of where they live. Give it a title that states what you found.
3. The `Nominations` column holds numbers, so we can compare it with `>=` instead of `==`. Make a filtered DataFrame of the authors with 3 or more nominations, and print their names and nominations. (Hint: `filtered[["Author", "Nominations"]]` picks two columns at once.)
4. **Challenge:** Who was born in Canada but lives elsewhere? You'll need two conditions at once. In pandas, `&` plays the role of `and`, and each condition needs its own parentheses:
   ```python
   authors[(condition 1) & (condition 2)]
   ```